# 05 — Executable Dependency Analysis


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

from build_optimiser.config import Config
from build_optimiser.features import (
    compute_exe_library_matrix,
    compute_jaccard_matrix,
    detect_thin_dependencies,
    identify_core_libraries,
)
from build_optimiser.graph import attach_metrics, load_graph

cfg = Config.from_yaml('../../config.yaml')
file_df = pd.read_parquet(cfg.processed_data_dir / 'file_metrics.parquet')
target_df = pd.read_parquet(cfg.processed_data_dir / 'target_metrics.parquet')
edge_df = pd.read_parquet(cfg.processed_data_dir / 'edge_list.parquet')
G = load_graph(cfg.processed_data_dir / 'edge_list.parquet')
attach_metrics(G, target_df)

results_dir = Path('../data/results')
results_dir.mkdir(parents=True, exist_ok=True)


## 1. Dependency Closure Computation

Build the executable × library reachability matrix. Each cell is 1 if the library is in the transitive closure of the executable's dependencies.


In [ ]:
# Prepare target type lookup for compute_exe_library_matrix.
target_types = target_df[['cmake_target', 'target_type']].copy()

# compute_exe_library_matrix returns long format (executable, library, is_direct).
# Keep long format for function calls; pivot to wide binary matrix for analysis.
exe_lib_long = compute_exe_library_matrix(G, target_types)

if len(exe_lib_long) > 0:
    exe_lib_matrix = exe_lib_long.pivot_table(
        index='executable', columns='library', values='is_direct',
        aggfunc='max', fill_value=False,
    ).astype(int)
else:
    exe_lib_matrix = pd.DataFrame()

print(f'Executables:  {exe_lib_matrix.shape[0]}')
print(f'Libraries:    {exe_lib_matrix.shape[1]}')
if exe_lib_matrix.size > 0:
    print(f'Density:      {exe_lib_matrix.values.mean():.2%}')

    # Distribution of closure sizes (number of libraries per executable).
    closure_sizes = exe_lib_matrix.sum(axis=1).sort_values(ascending=False)
    print()
    print('Closure size distribution:')
    print(closure_sizes.describe())
    print()
    print('Largest closures:')
    print(closure_sizes.head(15).to_string())
else:
    closure_sizes = pd.Series(dtype=float)
    print('No executable-library dependencies found.')

In [ ]:
# Most-depended-on libraries: how many executables include this library in their closure.
lib_reach = exe_lib_matrix.sum(axis=0).sort_values(ascending=False)
n_exes = exe_lib_matrix.shape[0]
lib_reach_df = lib_reach.reset_index()
lib_reach_df.columns = ['library', 'exe_count']
lib_reach_df['reach_pct'] = lib_reach_df['exe_count'] / n_exes

print('Most-depended-on libraries:')
print(lib_reach_df.head(20).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(closure_sizes.values, bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Libraries in transitive closure')
axes[0].set_ylabel('Executable count')
axes[0].set_title('Closure size distribution')

top_n = min(20, len(lib_reach_df))
axes[1].barh(lib_reach_df.head(top_n)['library'], lib_reach_df.head(top_n)['reach_pct'], color='steelblue')
axes[1].set_xlabel('Fraction of executables depending on library')
axes[1].set_title(f'Top {top_n} most-reached libraries')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
# Save exe_library_matrix in long format for use in notebook 06.
# Functions like identify_core_libraries and compute_jaccard_matrix expect long format.
exe_lib_long.to_parquet(cfg.processed_data_dir / 'exe_library_matrix.parquet', index=False)
print(f'Saved exe_library_matrix.parquet ({len(exe_lib_long)} rows, long format)')

## 2. Jaccard Similarity Between Executables

Executables with similar dependency closures tend to belong to the same functional group. The Jaccard matrix captures this similarity.


In [ ]:
if len(exe_lib_long) > 0:
    jaccard_matrix = compute_jaccard_matrix(exe_lib_long)

    print(f'Jaccard matrix shape: {jaccard_matrix.shape}')
    if jaccard_matrix.shape[0] >= 2:
        print(f'Mean similarity: {jaccard_matrix.values[np.triu_indices_from(jaccard_matrix.values, k=1)].mean():.3f}')
    else:
        print('Only one executable — cannot compute pairwise similarity.')
else:
    jaccard_matrix = pd.DataFrame()
    print('No exe-library edges — skipping Jaccard.')

In [ ]:
# Clustered heatmap of Jaccard similarities.
# Limit to at most 60 executables to keep the heatmap readable.
n_plot = min(60, len(jaccard_matrix))
if len(jaccard_matrix) > n_plot:
    # Keep the executables with the largest closures (most interesting).
    top_exes = closure_sizes.head(n_plot).index.tolist()
    jac_plot = jaccard_matrix.loc[top_exes, top_exes]
    subtitle = f' (top {n_plot} by closure size)'
else:
    jac_plot = jaccard_matrix
    subtitle = ''

# Ensure diagonal is exactly 1 and matrix is symmetric.
jac_arr = jac_plot.values.copy()
np.fill_diagonal(jac_arr, 1.0)

g = sns.clustermap(
    jac_arr,
    method='average',
    metric='euclidean',
    xticklabels=jac_plot.columns.tolist(),
    yticklabels=jac_plot.index.tolist(),
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    figsize=(12, 12),
)
g.fig.suptitle(f'Jaccard similarity between executables{subtitle}', y=1.02, fontsize=13)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=6, rotation=90)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=6)
plt.show()


## 3. Core Library Identification

Libraries depended on by a large fraction of executables are *core* — optimising them yields the widest build-time benefit.


In [ ]:
# Default threshold: a library is "core" if >= 50 % of executables depend on it.
core_threshold = 0.50
core_libraries = identify_core_libraries(exe_lib_long, threshold=core_threshold)

print(f'Core libraries (threshold {core_threshold:.0%}): {len(core_libraries)}')
print()
print(sorted(core_libraries))

In [ ]:
# Frequency distribution of how many executables depend on each library.
fig, ax = plt.subplots(figsize=(10, 5))

reach_pcts = lib_reach_df['reach_pct'].values
ax.hist(reach_pcts, bins=20, color='steelblue', edgecolor='white', label='All libraries')
ax.axvline(core_threshold, color='red', linestyle='--', label=f'Core threshold ({core_threshold:.0%})')

# Annotate the core libraries near the top of the chart.
core_reaches = lib_reach_df[lib_reach_df['library'].isin(core_libraries)]['reach_pct']
ax.scatter(
    core_reaches,
    np.full(len(core_reaches), ax.get_ylim()[1] * 0.95),
    marker='v',
    color='red',
    zorder=5,
    label='Core library',
)

ax.set_xlabel('Fraction of executables depending on library')
ax.set_ylabel('Library count')
ax.set_title('Library reach frequency distribution')
ax.legend()
plt.tight_layout()
plt.show()

# Compile time share of core libraries.
core_ct = target_df[target_df['cmake_target'].isin(core_libraries)]['compile_time_sum_ms'].sum()
total_ct = target_df['compile_time_sum_ms'].sum()
print(f'Core libraries compile time share: {core_ct / total_ct:.1%}')


## 4. Thin Dependency Detection

A *thin dependency* is an edge where the dependent target uses very little of the dependency's public interface — often a sign of over-broad linking or header leakage that can be severed to reduce rebuild amplification.


In [ ]:
# detect_thin_dependencies expects a DataFrame with (source_file, cmake_target, header_tree).
# Use file_df directly if it has the required columns.
if 'header_tree' in file_df.columns and 'cmake_target' in file_df.columns:
    header_data = file_df[['source_file', 'cmake_target', 'header_tree']].copy()
else:
    header_data = pd.DataFrame(columns=['source_file', 'cmake_target', 'header_tree'])

thinness_threshold = 0.05  # < 5 % header overlap => thin
thin_deps = detect_thin_dependencies(G, header_data, thinness_threshold=thinness_threshold)

thin_df = pd.DataFrame(thin_deps) if isinstance(thin_deps, list) else thin_deps
print(f'Thin dependencies detected: {len(thin_df)}')
if not thin_df.empty:
    print(thin_df.head(20).to_string(index=False))

In [ ]:
# Save thin dependencies report.
thin_df.to_csv(results_dir / 'thin_dependencies.csv', index=False)
print(f'Saved thin_dependencies.csv ({len(thin_df)} rows)')


## 5. Executable Grouping Preview

Hierarchical clustering of executables using the Jaccard distance matrix. Cut the dendrogram at several levels to preview natural groupings before feature group discovery in notebook 06.


In [ ]:
# Convert Jaccard similarity to distance, then cluster.
jac_full = jaccard_matrix.values.copy()
np.fill_diagonal(jac_full, 1.0)
dist_mat = 1.0 - jac_full
# Ensure symmetry and non-negative.
dist_mat = (dist_mat + dist_mat.T) / 2
np.fill_diagonal(dist_mat, 0.0)
dist_mat = np.clip(dist_mat, 0, None)

condensed = squareform(dist_mat)
Z = linkage(condensed, method='average')

fig, ax = plt.subplots(figsize=(16, 6))
dend = dendrogram(
    Z,
    labels=jaccard_matrix.index.tolist(),
    leaf_rotation=90,
    leaf_font_size=7,
    ax=ax,
    color_threshold=0.5,
)
ax.set_title('Hierarchical clustering of executables by dependency overlap (Jaccard distance)')
ax.set_xlabel('Executable')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()


In [ ]:
from scipy.cluster.hierarchy import fcluster

# Cut at multiple distance thresholds and report group structure.
exe_names = jaccard_matrix.index.tolist()
print(f'{'Threshold':>10}  {'Groups':>6}  {'Max group':>10}  {'Min group':>10}')
print('-' * 44)
for threshold in [0.3, 0.5, 0.7, 0.9]:
    labels = fcluster(Z, t=threshold, criterion='distance')
    group_sizes = pd.Series(labels).value_counts()
    print(f'{threshold:>10.1f}  {len(group_sizes):>6}  {group_sizes.max():>10}  {group_sizes.min():>10}')

# Detailed view at the midpoint threshold.
mid_threshold = 0.5
labels_mid = fcluster(Z, t=mid_threshold, criterion='distance')
group_df = pd.DataFrame({'executable': exe_names, 'group_id': labels_mid})
print()
print(f'Groups at threshold {mid_threshold}:')
for gid, members in group_df.groupby('group_id')['executable']:
    member_list = members.tolist()
    print(f'  Group {gid:2d} ({len(member_list):3d} members): {member_list[:5]}{"..." if len(member_list) > 5 else ""}')
